# Atividade 2

## Variáveis de Sistema

In [0]:
#widgets
dbutils.widgets.text("data_inicial", "2026-01-01","(YYYY-MM-DD)")
dbutils.widgets.text("data_final","2026-01-31","(YYYY-MM-DD)")

In [0]:
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

In [0]:
%sql
USE CATALOG cine_medallion;
USE SCHEMA default;

-- TODO: List the files in the myfiles volume
LIST "/Volumes/cine_medallion/bronze/landing/";

In [0]:
from pyspark.sql.functions import current_timestamp
#paths no volume ,estabelecidos no notebook 00 landing
path_credits = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_finacials= f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_info= f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_metrics= f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_reviews= f"{landing_path}/movies_reviews.csv"

df_credits_raw = spark.read.csv(path_credits, header=True, inferSchema=True)
df_financials_raw  = spark.read.csv(path_finacials, header=True, inferSchema=True)
df_info_raw =  spark.read.csv(path_info, header=True, inferSchema=True)
df_metrics_raw  = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_reviews_raw  = spark.read.csv(path_reviews, header=True, inferSchema=True)

In [0]:
#reading the data frame
tables = []
df_credits_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.credits")

df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.financials")   

df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.info")

df_metrics_raw\
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.metrics")   
    
df_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.reviews")

In [0]:
import requests
import pyspark.sql.functions as F
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo("
    "dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)
#paramns 
from datetime import datetime
data_inicial = datetime.strptime(
    dbutils.widgets.get("data_inicial").strip(), "%Y-%m-%d"
).date()
data_final= datetime.strptime(
    dbutils.widgets.get("data_final").strip(), "%Y-%m-%d"
).date()
if data_inicial > data_final:
    raise ValueError("data inicial maior que a data final.")
params ={
    "@dataInicial": f"'{data_inicial:%m-%d-%Y}'",
    "@dataFinalCotacao": f"'{data_final:%m-%d-%Y}'",
    "$format": "json",
    "$top": 1000,
    "$skip": 0
}
dados = []
with requests.Session() as session:
    while True:
        resposta = session.get(url, params = params, timeout = 60)
        resposta.raise_for_status()

        pagina = resposta.json()["value"]
        if not pagina:
            break
        dados.extend(pagina)
        params["$skip"] += len(pagina)

df_dolar_raw = spark.createDataFrame(
    dados,
    schema ="""
            cotacaoCompra DOUBLE,
            cotacaoVenda DOUBLE,
            dataHoraCotacao STRING

    """
)
df_dolar_raw = (
    df_dolar_raw
    .withColumn("cotacaoCompra", F.col("cotacaoCompra").cast("decimal(18,5)"))
    .withColumn("cotacaoVenda", F.col("cotacaoVenda").cast("decimal(18,5)"))
    .withColumn("dataHoraCotacao",F.to_timestamp("dataHoraCotacao"))
    .withColumn("data_ingestao", F.current_timestamp())

)
display(df_dolar_raw)

In [0]:
df_dolar_raw.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.dolar")